In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
import keras
keras.utils.set_random_seed(28)
import datetime
from icecream import ic
from itertools import product
from sklearn.preprocessing import MinMaxScaler
import wandb
from configparser import ConfigParser
from sklearn.model_selection import train_test_split

I0000 00:00:1776448071.371440  305242 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776448071.372176  305242 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776448071.423832  305242 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776448072.985195  305242 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

In [ ]:
config = ConfigParser()
config.read("credentials.ini")
wandb_api_key = config["WandB"]["api_key"]

In [3]:
CUSTOM_DATASET_PATH = Path("custom_data/")

In [4]:
merged = pd.read_csv(CUSTOM_DATASET_PATH / "merged.csv")
merged

,business_id,latitude,longitude,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
1,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
2,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,3,8,2,2,5,0,18
3,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
4,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
511134,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
511135,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
511136,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


In [5]:
weekdays = ("Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday")
merged = merged.drop(columns=[weekday.lower() + suffix for weekday, suffix in product(weekdays, ("_opening", "_closing", "_opening_length_hours"))])


In [6]:
merged = merged.drop(columns=["latitude", "longitude"])

In [7]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(merged.isnull().sum())

business_id                                         0
stars_business                                      0
review_count_review                                 0
Fuzhou_category                                     0
Themed Cafes_category                               0
Burmese_category                                    0
Falafel_category                                    0
Cafes_category                                      0
Conveyor Belt Sushi_category                        0
Australian_category                                 0
Sushi Bars_category                                 0
Mongolian_category                                  0
New Mexican Cuisine_category                        0
Poutineries_category                                0
Ethiopian_category                                  0
Hong Kong Style Cafe_category                       0
Cafeteria_category                                  0
Chicken Shop_category                               0
Vegetarian_category         

In [8]:
merged.info(max_cols=500)

<class 'pandas.DataFrame'>
RangeIndex: 511138 entries, 0 to 511137
Data columns (total 388 columns):
 #    Column                                            Non-Null Count   Dtype  
---   ------                                            --------------   -----  
 0    business_id                                       511138 non-null  str    
 1    stars_business                                    511138 non-null  float64
 2    review_count_review                               511138 non-null  int64  
 3    Fuzhou_category                                   511138 non-null  int64  
 4    Themed Cafes_category                             511138 non-null  int64  
 5    Burmese_category                                  511138 non-null  int64  
 6    Falafel_category                                  511138 non-null  int64  
 7    Cafes_category                                    511138 non-null  int64  
 8    Conveyor Belt Sushi_category                      511138 non-null  int64  
 9    Au

In [9]:
timestamp_format = "%Y-%m-%d %H:%M:%S"
merged["date"] = merged["date"].apply(lambda x: datetime.datetime.strptime(x, timestamp_format))
merged["yelping_since"] = merged["yelping_since"].apply(lambda x: datetime.datetime.strptime(x, timestamp_format))
merged["date"] = (merged["date"] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')
merged["yelping_since"] = (merged["yelping_since"] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')

In [10]:
merged_no_ids = merged.drop(labels=["review_id", "user_id", "business_id"], axis=1)
display(merged_no_ids)

,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,Conveyor Belt Sushi_category,Australian_category,Sushi Bars_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,4.0,80,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
1,4.0,80,0,0,0,0,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
2,4.0,80,0,0,0,0,0,0,0,0,...,0,0,0,3,8,2,2,5,0,18
3,4.0,80,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
4,4.0,80,0,0,0,0,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,4.5,35,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
511134,4.5,35,0,0,0,0,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
511135,4.5,35,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
511136,4.5,35,0,0,0,0,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


In [11]:
merged_only_ids = merged[["review_id", "user_id", "business_id"]]
merged_only_ids

,review_id,user_id,business_id
0,BXQcBN0iAi1lAUxibGLFzA,6_SpY41LIHZuIaiDs5FMKA,MTSW4McQd7CbVtyjqoe9mw
1,uduvUCvi9w3T2bSGivCfXg,tCXElwhzekJEH6QJe3xs7Q,MTSW4McQd7CbVtyjqoe9mw
2,a0vwPOqDXXZuJkbBW2356g,WqfKtI-aGMmvbA9pPUxNQQ,MTSW4McQd7CbVtyjqoe9mw
3,MKNp_CdR2k2202-c8GN5Dw,3-1va0IQfK-9tUMzfHWfTA,MTSW4McQd7CbVtyjqoe9mw
4,D1GisLDPe84Rrk_R4X2brQ,EouCKoDfzaVG0klEgdDvCQ,MTSW4McQd7CbVtyjqoe9mw
...,...,...,...
511133,qBcwQEQPnLxjkw-xbUIF4Q,6nF5PT1c0dF6EpOgQdF2tw,WnT9NIzQgLlILjPT0kEcsQ
511134,G8fbysnUAUmqq1XWTjMQ4Q,1M78_w4J9f5S8xmUVYyxdQ,WnT9NIzQgLlILjPT0kEcsQ
511135,JKiy0aeyGd3KmXN7uRPFLw,B7TD5yTemGv50y4wM2EVNA,WnT9NIzQgLlILjPT0kEcsQ
511136,JITY01bGbdsiUBznLz9rdg,HI8QwhpeP_ZRY5JZy11VDw,WnT9NIzQgLlILjPT0kEcsQ


In [12]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(merged_no_ids)
merged_no_ids = pd.DataFrame(scaler.transform(merged_no_ids), columns=merged_no_ids.columns)
display(merged_no_ids)

,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,Conveyor Belt Sushi_category,Australian_category,Sushi Bars_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000063,0.000000,0.000000
1,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000852,0.000336,0.000000,0.000186,0.000277,0.000921,0.000921,0.001632,0.000356,0.012338
2,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000079,0.000040,0.000040,0.000314,0.000000,0.001134
3,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000200
4,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000336,0.001534,0.000407,0.000504,0.000700,0.000700,0.000816,0.000089,0.009337
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000010,0.000000,0.000000,0.000000,0.000000,0.000000
511134,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000030,0.000080,0.000080,0.000063,0.000053,0.000333
511135,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
511136,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000085,0.000000,0.000000,0.000000,0.000000,0.000000,0.005069


In [13]:
merged_no_ids_copy = merged_no_ids.copy()
merged_no_ids_copy = merged_no_ids_copy.drop(columns=["review_count_user", "yelping_since", "useful_user", "funny_user", "cool_user", "fans", "average_stars", "compliment_hot", "compliment_more", "compliment_profile", "compliment_cute", "compliment_list", "compliment_note", "compliment_plain", "compliment_cool", "compliment_funny", "compliment_writer", "compliment_photos", "number_of_friends", "useful_review", "funny_review", "cool_review", "stars_review"])
merged_no_ids_copy

,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,Conveyor Belt Sushi_category,Australian_category,Sushi Bars_category,...,BestNightsThursday_attribute_False,BestNightsThursday_attribute_True,BestNightsThursday_attribute_no_info,DriveThru_attribute_False,DriveThru_attribute_True,DriveThru_attribute_no_info,BestNightsMonday_attribute_False,BestNightsMonday_attribute_True,BestNightsMonday_attribute_no_info,date
0,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.547730
1,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.510133
2,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.513276
3,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.783217
4,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.513282
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.933705
511134,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.932890
511135,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.951791
511136,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.943997


In [14]:
merged = merged_only_ids.join(merged_no_ids)
display(merged)

,review_id,user_id,business_id,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,BXQcBN0iAi1lAUxibGLFzA,6_SpY41LIHZuIaiDs5FMKA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000063,0.000000,0.000000
1,uduvUCvi9w3T2bSGivCfXg,tCXElwhzekJEH6QJe3xs7Q,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000852,0.000336,0.000000,0.000186,0.000277,0.000921,0.000921,0.001632,0.000356,0.012338
2,a0vwPOqDXXZuJkbBW2356g,WqfKtI-aGMmvbA9pPUxNQQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000079,0.000040,0.000040,0.000314,0.000000,0.001134
3,MKNp_CdR2k2202-c8GN5Dw,3-1va0IQfK-9tUMzfHWfTA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000200
4,D1GisLDPe84Rrk_R4X2brQ,EouCKoDfzaVG0klEgdDvCQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000336,0.001534,0.000407,0.000504,0.000700,0.000700,0.000816,0.000089,0.009337
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,qBcwQEQPnLxjkw-xbUIF4Q,6nF5PT1c0dF6EpOgQdF2tw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000010,0.000000,0.000000,0.000000,0.000000,0.000000
511134,G8fbysnUAUmqq1XWTjMQ4Q,1M78_w4J9f5S8xmUVYyxdQ,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000030,0.000080,0.000080,0.000063,0.000053,0.000333
511135,JKiy0aeyGd3KmXN7uRPFLw,B7TD5yTemGv50y4wM2EVNA,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
511136,JITY01bGbdsiUBznLz9rdg,HI8QwhpeP_ZRY5JZy11VDw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000085,0.000000,0.000000,0.000000,0.000000,0.000000,0.005069


In [15]:
merged_copy = merged_only_ids.join(merged_no_ids_copy)
display(merged_copy)

,review_id,user_id,business_id,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,...,BestNightsThursday_attribute_False,BestNightsThursday_attribute_True,BestNightsThursday_attribute_no_info,DriveThru_attribute_False,DriveThru_attribute_True,DriveThru_attribute_no_info,BestNightsMonday_attribute_False,BestNightsMonday_attribute_True,BestNightsMonday_attribute_no_info,date
0,BXQcBN0iAi1lAUxibGLFzA,6_SpY41LIHZuIaiDs5FMKA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.547730
1,uduvUCvi9w3T2bSGivCfXg,tCXElwhzekJEH6QJe3xs7Q,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.510133
2,a0vwPOqDXXZuJkbBW2356g,WqfKtI-aGMmvbA9pPUxNQQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.513276
3,MKNp_CdR2k2202-c8GN5Dw,3-1va0IQfK-9tUMzfHWfTA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.783217
4,D1GisLDPe84Rrk_R4X2brQ,EouCKoDfzaVG0klEgdDvCQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.513282
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,qBcwQEQPnLxjkw-xbUIF4Q,6nF5PT1c0dF6EpOgQdF2tw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.933705
511134,G8fbysnUAUmqq1XWTjMQ4Q,1M78_w4J9f5S8xmUVYyxdQ,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.932890
511135,JKiy0aeyGd3KmXN7uRPFLw,B7TD5yTemGv50y4wM2EVNA,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.951791
511136,JITY01bGbdsiUBznLz9rdg,HI8QwhpeP_ZRY5JZy11VDw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.943997


In [16]:
sequence_length = 5

In [17]:
indexes = merged.value_counts("user_id")[(merged.value_counts("user_id") >= sequence_length)].index
indexes

Index(['ET8n-r7glWYqZhuR6GcdNw', 'vFd8aBLg1kFcd0kCkoi-xw',
       '_BcWyKQL16ndpBdggh2kNA', 'bJ5FtCtZX3ZZacz2_2PJjA',
       '0DB3Irpf_ETVXu_Ou9vPow', 'kauJmG3ZiA-m5u0nPrjb4g',
       'WO6L5yMX5LEeJuMNMjerRQ', '8EMU7d4pCkdqUnvlIW40CA',
       'LnFIWZM_l__4t8Qxj3pnOg', 'OLGQ7alK4VKl3YdQk6UF5g',
       ...
       'GQgH--hxKJisGmfsKj0xDA', 'Zy8NOrdYcTaKJxbfdM0kHQ',
       'nh6ArLduHi_53eMPMW0fgw', 'OxaqY6xoDhFFxyICEX4QQQ',
       'EWLuooWEKiGZlxl3jccVyA', 'LU0KeX-SHEFhI77xqDueoA',
       'MKkCT6R15R2nrURE6VTOgA', 'kJModxtvXak455L9xo9xXw',
       'RhoZYeIa3tP0nTEitoNFzA', 'knE5x-gQ6ALjcXHpSkpkVw'],
      dtype='str', name='user_id', length=19902)

In [18]:
def split_to_sequences(df, user_indices, sequence_length):
    user_sequences = []
    for user_id in user_indices:
        user_df = df[df["user_id"] == user_id].sort_values("date")
        # print("zaciatok", len(user_df))

        while len(user_df) >= sequence_length:
            user_sequences.append(user_df.iloc[:sequence_length])
            user_df = user_df.iloc[sequence_length:]
        # print("end", len(user_df))
    return user_sequences

In [20]:
user_sequences = split_to_sequences(merged, user_indices=indexes, sequence_length=5)
print("user_sequences split")
user_sequences_copy = split_to_sequences(merged_copy, user_indices=indexes, sequence_length=5)
print("user_sequences_copy split")

user_sequences split
user_sequences_copy split


In [ ]:
# user_sequences = []
# for user_id in indexes:
#     user_df = merged[merged["user_id"] == user_id].sort_values("date")
#     # print("zaciatok", len(user_df))
# 
#     while len(user_df) >= sequence_length:
#         user_sequences.append(user_df.iloc[:sequence_length])
#         user_df = user_df.iloc[sequence_length:]
#     # print("end", len(user_df))

zaciatok 610
end 0
zaciatok 447
end 2
zaciatok 441
end 1
zaciatok 375
end 0
zaciatok 357
end 2
zaciatok 343
end 3
zaciatok 332
end 2
zaciatok 331
end 1
zaciatok 277
end 2
zaciatok 272
end 2
zaciatok 271
end 1
zaciatok 261
end 1
zaciatok 260
end 0
zaciatok 260
end 0
zaciatok 246
end 1
zaciatok 241
end 1
zaciatok 229
end 4
zaciatok 229
end 4
zaciatok 227
end 2
zaciatok 226
end 1
zaciatok 225
end 0
zaciatok 220
end 0
zaciatok 215
end 0
zaciatok 214
end 4
zaciatok 211
end 1
zaciatok 203
end 3
zaciatok 200
end 0
zaciatok 200
end 0
zaciatok 199
end 4
zaciatok 197
end 2
zaciatok 197
end 2
zaciatok 196
end 1
zaciatok 195
end 0
zaciatok 181
end 1
zaciatok 178
end 3
zaciatok 178
end 3
zaciatok 176
end 1
zaciatok 176
end 1
zaciatok 172
end 2
zaciatok 170
end 0
zaciatok 170
end 0
zaciatok 169
end 4
zaciatok 167
end 2
zaciatok 166
end 1
zaciatok 166
end 1
zaciatok 166
end 1
zaciatok 165
end 0
zaciatok 163
end 3
zaciatok 161
end 1
zaciatok 161
end 1
zaciatok 159
end 4
zaciatok 159
end 4
zaciatok 159

In [53]:
user_sequences

[                     review_id                 user_id  \
 488637  ih0EwuprB0xNJLF3bugq4g  vFd8aBLg1kFcd0kCkoi-xw   
 329984  T5Su0cLipsEmqA0aq6BAVw  vFd8aBLg1kFcd0kCkoi-xw   
 445552  HPGzJv1vZIpUjgfT7FtygQ  vFd8aBLg1kFcd0kCkoi-xw   
 203384  8BdCkdxsTdZV07kh8oovQg  vFd8aBLg1kFcd0kCkoi-xw   
 368911  P1Ky8dZB5ECiCRFcJZqSfA  vFd8aBLg1kFcd0kCkoi-xw   
 
                    business_id  stars_business  review_count_review  \
 488637  -1B9pP_CrRBJYPICE5WbRA           0.750             0.142932   
 329984  laXWQMgAW2kUfAH7Y-xH8w           0.875             0.020119   
 445552  atZ_olNKXOG4rEr6mccN8g           0.875             0.238453   
 203384  oziGQYLaEWOpNmPKrZP86g           0.875             0.081700   
 368911  t6g6Lj8WzjIXXucCl9jdTQ           0.750             0.069279   
 
         Fuzhou_category  Themed Cafes_category  Burmese_category  \
 488637              0.0                    0.0               0.0   
 329984              0.0                    0.0               0.0   
 44

In [21]:
len(user_sequences)

49560

In [22]:
len(user_sequences_copy)

49560

In [ ]:
# merged

,review_id,user_id,business_id,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,BXQcBN0iAi1lAUxibGLFzA,6_SpY41LIHZuIaiDs5FMKA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000063,0.000000,0.000000
1,uduvUCvi9w3T2bSGivCfXg,tCXElwhzekJEH6QJe3xs7Q,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000852,0.000336,0.000000,0.000186,0.000277,0.000921,0.000921,0.001632,0.000356,0.012338
2,a0vwPOqDXXZuJkbBW2356g,WqfKtI-aGMmvbA9pPUxNQQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000079,0.000040,0.000040,0.000314,0.000000,0.001134
3,MKNp_CdR2k2202-c8GN5Dw,3-1va0IQfK-9tUMzfHWfTA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000200
4,D1GisLDPe84Rrk_R4X2brQ,EouCKoDfzaVG0klEgdDvCQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000336,0.001534,0.000407,0.000504,0.000700,0.000700,0.000816,0.000089,0.009337
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,qBcwQEQPnLxjkw-xbUIF4Q,6nF5PT1c0dF6EpOgQdF2tw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000010,0.000000,0.000000,0.000000,0.000000,0.000000
511134,G8fbysnUAUmqq1XWTjMQ4Q,1M78_w4J9f5S8xmUVYyxdQ,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000030,0.000080,0.000080,0.000063,0.000053,0.000333
511135,JKiy0aeyGd3KmXN7uRPFLw,B7TD5yTemGv50y4wM2EVNA,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
511136,JITY01bGbdsiUBznLz9rdg,HI8QwhpeP_ZRY5JZy11VDw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000085,0.000000,0.000000,0.000000,0.000000,0.000000,0.005069


In [23]:
user_matrix = np.array(
    [user_df.drop(columns=["review_id", "user_id", "business_id"]).to_numpy() for user_df in user_sequences]
)

In [24]:
user_matrix_y = np.array(
    [user_df.drop(columns=["review_id", "user_id", "business_id"]).to_numpy() for user_df in user_sequences_copy]
)

In [26]:
display(user_matrix.shape)
display(user_matrix.dtype)

(49560, 5, 385)

dtype('float64')

In [27]:
display(user_matrix_y.shape)
display(user_matrix_y.dtype)

(49560, 5, 362)

dtype('float64')

In [ ]:
# user_matrix = np.array(
#     [merged[merged["user_id"] == user_id].sort_values("date").drop(labels=["review_id", "user_id", "business_id"], axis=1).to_numpy() for user_id in indexes]
# )

In [ ]:
# user_matrix_y = np.array(
#     [merged_copy[merged_copy["user_id"] == user_id].sort_values("date").drop(labels=["review_id", "user_id", "business_id", "date"], axis=1).to_numpy() for user_id in indexes]
# )

In [28]:
X_train, X_test, y_train, y_test = train_test_split(user_matrix, user_matrix_y, train_size=0.8, random_state=28)

In [29]:
display(X_train.shape)
display(X_train.dtype)
display(X_test.shape)
display(X_test.dtype)
display(y_train.shape)
display(y_train.dtype)
display(y_test.shape)
display(y_test.dtype)

(39648, 5, 385)

dtype('float64')

(9912, 5, 385)

dtype('float64')

(39648, 5, 362)

dtype('float64')

(9912, 5, 362)

dtype('float64')

In [ ]:
X_train = X_train[:, :sequence_length-1, :]
X_test = X_test[:, :sequence_length-1, :]
y_train = y_train[:, sequence_length-1:, :]
y_test = y_test[:, sequence_length-1:, :]

In [31]:
display(X_train.shape)
display(X_test.shape)
display(y_train.shape)
display(y_test.shape)

(39648, 4, 385)

(9912, 4, 385)

(39648, 1, 362)

(9912, 1, 362)

In [32]:
display(0.1/(32*4*385))
display(1/(32*4*385))
display((0.1/(32*4*385)) + (1/(32*4*385)))

2.0292207792207792e-06

2.0292207792207792e-05

2.232142857142857e-05

In [33]:
# hyperparameters and stuff
learning_rate = 0.000001
number_of_epochs = 300
batch_size = 32

In [34]:
class ModelCallback(keras.callbacks.Callback):
    def __init__(self, X_test, y_test, batch_size):
        self.X_test = X_test
        self.y_test = y_test
        self.batch_size = batch_size

    def on_epoch_end(self, epoch, logs=None):
        new_logs = dict()
        for metric in logs:
            new_logs["train_" + metric] = logs[metric]
        evaluation = self.model.evaluate(x=self.X_test, y=self.y_test, batch_size=self.batch_size)
        loss = evaluation[0]
        mae = evaluation[1]
        new_logs["test_loss"] = loss
        new_logs["test_mae"] = mae
        run.log(new_logs)

    def on_train_end(self, logs=None):
        pass

    def on_epoch_begin(self, epoch, logs=None):
        pass

    def on_test_begin(self, logs=None):
        pass

    def on_test_end(self, logs=None):
        pass

In [39]:
model = Sequential([
    LSTM(512, activation="tanh", return_sequences=True, input_shape=(sequence_length-1, 385)),
    LSTM(384, activation="tanh"),
    Dense(256, activation="relu"),
    # Dropout(0.1),
    Dense(362, activation="tanh"), # nejako to rozdelit na ratingy a binarne featury
])
optimizer = keras.optimizers.Adam()
optimizer.learning_rate.assign(learning_rate)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
model.summary()

/home/kristofs/School/ISA/ISA_project/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 4, 512)         │     1,839,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 384)            │     1,377,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 362)            │        93,034 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,408,490 (13.00 MB)

 Trainable params: 3,408,490 (13.00 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
model_callback = ModelCallback(
    X_test=X_test,
    y_test=y_test,
    batch_size=batch_size
)

In [37]:
run = wandb.init(
    entity="ISA2026",
    project="hello_world",
    name="test_run",
    config={
        "learning_rate": learning_rate,
        "number_of_epochs": number_of_epochs,
        "batch_size": batch_size,
    },
)
run.log_code(".", include_fn=lambda path: path.endswith(".ipynb"))

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/kristofs/.netrc.
wandb: Currently logged in as: stanislavkristof to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


<Artifact source-hello_world-_home_kristofs_School_ISA_ISA_project_lstm.ipynb>

In [41]:
history = model.fit(X_train, y_train, epochs=number_of_epochs, batch_size=batch_size, verbose=2, callbacks=[model_callback])

Epoch 1/300
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1655 - mae: 0.2565
1239/1239 - 28s - 22ms/step - loss: 0.1916 - mae: 0.2435
Epoch 2/300
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1191 - mae: 0.2380
1239/1239 - 26s - 21ms/step - loss: 0.1399 - mae: 0.2517
Epoch 3/300
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0950 - mae: 0.2022
1239/1239 - 26s - 21ms/step - loss: 0.1060 - mae: 0.2200
Epoch 4/300
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0823 - mae: 0.1758
1239/1239 - 25s - 20ms/step - loss: 0.0881 - mae: 0.1883
Epoch 5/300
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0774 - mae: 0.1608
1239/1239 - 25s - 20ms/step - loss: 0.0796 - mae: 0.1674
Epoch 6/300
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0760 - mae: 0.1559
1239/1239 - 25s - 20ms/step - loss: 0.0768 - mae: 0.1579
Epoch 7/300
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0754 - mae: 0.1544
1239/1239 - 25s - 20ms/step - loss: 0.0759 - mae: 0.1551
Epoch 8/300
310/310 ━━━━━━━

In [ ]:
model.evaluate(x=X_test, y=y_test, batch_size=batch_size, verbose=2)

In [39]:
run.finish()

test_loss,█▆▄▁
test_mae,▃▁▃█
train_loss,█▆▄▁
train_mae,█▁▂▇
test_loss,0.19814
test_mae,0.23517
train_loss,0.20031
train_mae,0.23455
